# claude implemented striped mamba, let's test it

# the plan

# Striped Mamba Backbone with U-Net Sampling

## Context
Build a striped mamba model that combines Hydra (bidirectional SSM) and gated MHA in a U-Net style architecture with configurable up/downsampling. The existing Hydra and MHAGate modules are already implemented but need to be wrapped in full blocks (norm + residual + MLP). The up/downsampling currently lives in the encoder/decoder but needs to move into the backbone itself, with simplified encoder/decoder that just do projection.

## New Files

### 1. `/data1/lesliec/sarthak/caduceus/src/models/nn/blocks.py` — Full Blocks

**HydraBlock** — wraps Hydra with pre-norm residual + MLP (like enigma's TransformerBlock)
```python
class HydraBlock(nn.Module):
    def __init__(self, d_model, d_state=64, d_conv=7, expand=2, headdim=64,
                 expansion_factor=4, norm='ln', dropout=0.0, **hydra_kwargs):
        self.mixer_norm = LayerNorm/RMSNorm(d_model)
        self.mixer = Hydra(d_model, d_state, d_conv, expand, headdim, **hydra_kwargs)
        self.mixer_dropout = nn.Dropout(dropout)
        self.mlp_norm = LayerNorm/RMSNorm(d_model)
        self.mlp = MLP(d_model, d_model * expansion_factor)  # Linear→Act→Linear
        self.mlp_dropout = nn.Dropout(dropout)

    def forward(self, x):  # (B, L, d_model) → (B, L, d_model)
        h = x + self.mixer_dropout(self.mixer(self.mixer_norm(x)))
        out = h + self.mlp_dropout(self.mlp(self.mlp_norm(h)))
        return out
```

**TransformerBlock** — wraps MHAGate with pre-norm residual + MLP
```python
class TransformerBlock(nn.Module):
    def __init__(self, d_model, head_dim=64, expansion_factor=4, norm='ln',
                 dropout=0.0, use_rope=True, use_flash_attn=True, **mha_kwargs):
        self.attn_norm = LayerNorm/RMSNorm(d_model)
        self.attn = MHAGate(d_model, head_dim, use_rope=use_rope,
                            use_flash_attn=use_flash_attn, **mha_kwargs)
        self.post_attn_dropout = nn.Dropout(dropout)
        self.mlp_norm = LayerNorm/RMSNorm(d_model)
        self.mlp = MLP(d_model, d_model * expansion_factor)
        self.post_mlp_dropout = nn.Dropout(dropout)

    def forward(self, x):  # (B, L, d_model) → (B, L, d_model)
        h = x + self.post_attn_dropout(self.attn(self.attn_norm(x)))
        out = h + self.post_mlp_dropout(self.mlp(self.mlp_norm(h)))
        return out
```

**MLP** — simple feed-forward (reuse enigma pattern)
```python
class MLP(nn.Module):
    def __init__(self, d_model, d_hidden, activation='gelu', dropout=0.0):
        self.fc1 = nn.Linear(d_model, d_hidden)
        self.act = activation_fn
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        return self.fc2(self.dropout(self.act(self.fc1(x))))
```

### 2. `/data1/lesliec/sarthak/caduceus/src/models/nn/sampling.py` — Up/Downsample Modules

**DownsampleStack** — log2(factor) CNN downsample layers, stores intermediates
```python
class DownsampleStack(nn.Module):
    def __init__(self, d_model, factor):
        # factor must be power of 2
        n_layers = int(log2(factor))
        # Each layer: Conv1d(d_model, d_model, kernel_size=4, stride=2, padding=1)
        # This halves sequence length while keeping d_model constant
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(d_model, d_model, kernel_size=4, stride=2, padding=1),
                nn.LayerNorm(d_model),  # applied after permute
                nn.GELU()
            ) for _ in range(n_layers)
        ])

    def forward(self, x):  # (B, L, d_model) → (B, L//factor, d_model), intermediates
        intermediates = []
        for layer in self.layers:
            intermediates.append(x)  # store pre-downsample for skip connection
            x = x.transpose(1, 2)    # (B, d_model, L)
            x = layer[0](x)          # conv: (B, d_model, L//2)
            x = x.transpose(1, 2)    # (B, L//2, d_model)
            x = layer[1](x)          # norm
            x = layer[2](x)          # activation
        return x, intermediates
```

**UpsampleStack** — log2(factor) CNN upsample layers with UNET skip connections
```python
class UpsampleStack(nn.Module):
    def __init__(self, d_model, factor):
        n_layers = int(log2(factor))
        # Each layer: ConvTranspose1d to double seq length
        # Skip connection: concatenate then project back to d_model
        self.layers = nn.ModuleList([
            nn.Sequential(
                nn.ConvTranspose1d(d_model, d_model, kernel_size=4, stride=2, padding=1),
                nn.LayerNorm(d_model),
                nn.GELU()
            ) for _ in range(n_layers)
        ])
    def forward(self, x, intermediates):
        # intermediates is reversed list from DownsampleStack
        for i, layer in enumerate(self.layers):
            x = x.transpose(1, 2)     # (B, d_model, L)
            x = layer[0](x)           # conv_transpose: (B, d_model, L*2)
            x = x.transpose(1, 2)     # (B, L*2, d_model)
            x = layer[1](x)           # norm
            x = layer[2](x)           # activation
            skip = intermediates[-(i+1)]  # UNET: pair with matching scale
            x = x + skip              # element-wise add (simple, same dims)
        return x
```

### 3. `/data1/lesliec/sarthak/caduceus/src/models/sequence/striped_backbone.py` — Main Backbone

```python
class StripedMambaBackbone(nn.Module):
    def __init__(self, d_model, n_blocks, global_pooling=1, transformer_pooling=1,
                 mode='striped',  # 'striped' | 'transformer_only' | 'mamba_only'
                 # Hydra config
                 d_state=64, d_conv=7, expand=2, headdim=64,
                 # Transformer config
                 head_dim=64, use_rope=True, use_flash_attn=True,
                 # Shared config
                 expansion_factor=4, norm='ln', dropout=0.0,
                 **kwargs):

        # Validate pooling factors are powers of 2
        assert global_pooling & (global_pooling - 1) == 0
        assert transformer_pooling & (transformer_pooling - 1) == 0

        self.mode = mode
        self.global_pooling = global_pooling
        self.transformer_pooling = transformer_pooling

        # --- Global downsample/upsample (if global_pooling > 1) ---
        if global_pooling > 1:
            self.global_down = DownsampleStack(d_model, global_pooling)
            self.global_up = UpsampleStack(d_model, global_pooling)

        # --- Striped blocks ---
        self.hydra_blocks = nn.ModuleList()
        self.transformer_blocks = nn.ModuleList()
        self.trans_down = nn.ModuleList()
        self.trans_up = nn.ModuleList()

        for _ in range(n_blocks):
            if mode in ('striped', 'mamba_only'):
                self.hydra_blocks.append(
                    HydraBlock(d_model, d_state, d_conv, expand, headdim,
                               expansion_factor, norm, dropout))

            if mode in ('striped', 'transformer_only'):
                self.transformer_blocks.append(
                    TransformerBlock(d_model, head_dim, expansion_factor, norm,
                                    dropout, use_rope, use_flash_attn))

            if mode == 'striped' and transformer_pooling > 1:
                self.trans_down.append(DownsampleStack(d_model, transformer_pooling))
                self.trans_up.append(UpsampleStack(d_model, transformer_pooling))

        # Final norm
        self.final_norm = LayerNorm/RMSNorm(d_model)

    def forward(self, x):
        # x: (B, L, d_model) from encoder

        # 1) Global downsample
        global_intermediates = None
        if self.global_pooling > 1:
            x, global_intermediates = self.global_down(x)
            # x is now (B, L // global_pooling, d_model)

        # 2) Repeat n_blocks
        for i in range(n_blocks):
            if self.mode == 'striped':
                # a) Hydra block at pooled resolution
                x = self.hydra_blocks[i](x)
                # b) Downsample for transformer
                x, trans_intermediates = self.trans_down[i](x)
                # c) Transformer block at further compressed resolution
                x = self.transformer_blocks[i](x)
                # d) Upsample back with skip connections
                x = self.trans_up[i](x, trans_intermediates)

            elif self.mode == 'mamba_only':
                x = self.hydra_blocks[i](x)

            elif self.mode == 'transformer_only':
                x = self.transformer_blocks[i](x)

        # 3) Global upsample back to original length
        if self.global_pooling > 1:
            x = self.global_up(x, global_intermediates)

        x = self.final_norm(x)
        return x  # (B, L, d_model)
```

### 4. New Encoder in `/data1/lesliec/sarthak/caduceus/src/tasks/encoders.py`

**SimpleEncoder** — Conv1d projection only, no downsampling
```python
class SimpleEncoder(nn.Module):
    def __init__(self, d_model, d_input1=6, d_input2=None, joint=False,
                 kernel_size=15, norm=None):
        # Same conv logic as JointCNN but without any downsample/pool layers
        if joint:
            self.conv = Conv1d(d_input1 + d_input2, d_model, kernel_size, padding='same')
        else:
            self.conv1 = Conv1d(d_input1, d_model // 2, kernel_size, padding='same')
            self.conv2 = Conv1d(d_input2, d_model // 2, kernel_size, padding='same')
            self.combine = nn.Linear(d_model, d_model)
        # Normalization + ReLU

    def forward(self, x1, x2):
        # Apply conv(s) → norm → activation
        # Output: (B, L, d_model) — permuted for backbone input
        # No intermediates dict needed
```

### 5. New Decoder in `/data1/lesliec/sarthak/caduceus/src/tasks/decoders.py`

**SimpleDecoder** — Linear projection only, no upsampling
```python
class SimpleDecoder(nn.Module):
    def __init__(self, d_model, d_output1=5, d_output2=1):
        self.decoder1 = nn.Linear(d_model, d_output1)
        self.decoder2 = nn.Linear(d_model, d_output2)

    def forward(self, x, **kwargs):
        # x: (B, L, d_model)
        x1 = self.decoder1(x)  # sequence output
        x2 = F.softplus(self.decoder2(x))  # accessibility output
        return x1, x2
```

## Data Flow Example

seq_len=1024, d_model=256, global_pooling=4, transformer_pooling=8, n_blocks=3, mode='striped'

```
Encoder:  (B, 6, 1024) + (B, 2, 1024) → Conv → (B, 1024, 256)

Backbone:
  Global Down: (B, 1024, 256) → (B, 256, 256)  [store 2 intermediates]

  Block 1:
    HydraBlock:    (B, 256, 256) → (B, 256, 256)
    Trans Down:    (B, 256, 256) → (B, 32, 256)   [store 3 intermediates]
    TransBlock:    (B, 32, 256)  → (B, 32, 256)
    Trans Up:      (B, 32, 256)  → (B, 256, 256)  [using 3 intermediates]

  Block 2: same pattern
  Block 3: same pattern

  Global Up: (B, 256, 256) → (B, 1024, 256)  [using 2 intermediates]
  Final Norm: (B, 1024, 256)

Decoder:  (B, 1024, 256) → Linear → (B, 1024, 5) + (B, 1024, 1)
```

## Three Modes

| Mode | What runs per block | Pooling used |
|------|-------------------|--------------|
| `striped` | HydraBlock → downsample → TransformerBlock → upsample | global + transformer |
| `mamba_only` | HydraBlock only | global only (transformer_pooling ignored) |
| `transformer_only` | TransformerBlock only | global only (transformer_pooling ignored) |

## Files to Create/Modify

| File | Action |
|------|--------|
| `src/models/nn/blocks.py` | **CREATE** — HydraBlock, TransformerBlock, MLP |
| `src/models/nn/sampling.py` | **CREATE** — DownsampleStack, UpsampleStack |
| `src/models/sequence/striped_backbone.py` | **CREATE** — StripedMambaBackbone |
| `src/tasks/encoders.py` | **MODIFY** — add SimpleEncoder class |
| `src/tasks/decoders.py` | **MODIFY** — add SimpleDecoder class |

## Verification
1. Instantiate StripedMambaBackbone with small dims and verify forward pass shapes
2. Test all 3 modes (striped, mamba_only, transformer_only)
3. Test with global_pooling=1 (no global sampling) and >1
4. Test encoder → backbone → decoder full pipeline
5. Verify non-power-of-2 pooling raises assertion error


In [ ]:
#seems like it's actually quite good, but we need to verify this
#seems actually really straightforward, but we need to verify it follows similar logic to the DNA embedding
#then we need to verify downsample and upsample make sense and work as expected like alphagenome style
#then remove the simple encoder and decoder. Just same thing don't do pooling in them anymore!

import torch
import sys


In [ ]:
#first let's verify the up and downsampling works as expected
#ok immediately we see lots of issues, it doesn't do the growing channels, it doesn't do the convs and things like that.
#maybe we implemented it wrong?

#it's fine, just ask it to fix this

# verifying main block

In [1]:
#verifying that it follows the same structure of mamba blocks!
#one key difference is that we aren't using flash attention blocks...
#likely not a huge increase in speed, so it's ok. Can optimize for speed later
#but I implemented it anyways, now it should be more efficient? And should work for transformer + hydra. So we can simplify the code a lot
#also added checkpoints and optional fused mlp. So now we are good?
#added residual in fp32 which only works when striped and for mamba only actually won't work well
#but we didn't even have that used in the past
#also checkpointing is done per block not per mlp/attention, so less granular but easier
#maybe we should have bene using it tho... since we have a deep mamba model?
#final speedup is this fused add+norm, but let's not do it, too complicated lol


#otherwise I think it looks ok
#let's first test compile
import sys
import torch
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped')
# model = torch.compile(model)
model.cuda().bfloat16()

StripedMambaBackbone(
  (hydra_blocks): ModuleList(
    (0-3): 4 x HydraBlock(
      (mixer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mixer): Hydra(
        (in_proj): Linear(in_features=64, out_features=516, bias=False)
        (conv1d): Conv1d(384, 384, kernel_size=(7,), stride=(1,), padding=(3,), groups=384)
        (act): SiLU()
        (fc_D): Linear(in_features=128, out_features=2, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=128, out_features=64, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
      (mlp_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=64, out_features=256, bias=True)
        (fc2): Linear(in_features=256, out_features=64, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (act): GELU(approximate='none')
      )
      (mlp_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleL

In [2]:
#print number of parameters
print(sum(p.numel() for p in model.parameters()))

527512


In [3]:
import time                                                                                                                                                   

x = torch.randn(2, 1024, 64, device='cuda', dtype=torch.bfloat16)                                                                                               
model = model.cuda().bfloat16()
                                                                                                                                                                
# Warmup (compile happens on first call)                                                                                                                        
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

12.070427904836833 ms


In [4]:
#now let's compile it
model = torch.compile(model)
model.cuda().bfloat16()

OptimizedModule(
  (_orig_mod): StripedMambaBackbone(
    (hydra_blocks): ModuleList(
      (0-3): 4 x HydraBlock(
        (mixer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (mixer): Hydra(
          (in_proj): Linear(in_features=64, out_features=516, bias=False)
          (conv1d): Conv1d(384, 384, kernel_size=(7,), stride=(1,), padding=(3,), groups=384)
          (act): SiLU()
          (fc_D): Linear(in_features=128, out_features=2, bias=False)
          (norm): RMSNorm()
          (out_proj): Linear(in_features=128, out_features=64, bias=False)
        )
        (mixer_dropout): Dropout(p=0.0, inplace=False)
        (mlp_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=64, out_features=256, bias=True)
          (fc2): Linear(in_features=256, out_features=64, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
          (act): GELU(approximate='none')
        )
        (mlp_dropout):

In [5]:
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

[2026-03-28 10:28:42,669] [0/0] torch._dynamo.variables.higher_order_ops: [WARNING] speculate_subgraph: while introspecting the user-defined autograd.Function, we were unable to trace function `trampoline_autograd_fwd` into a single graph. This means that Dynamo was unable to prove safety for this API and will fall back to eager-mode PyTorch, which could lead to a slowdown.
[2026-03-28 10:28:42,670] [0/0] torch._dynamo.variables.higher_order_ops: [ERROR] HigherOrderOperator with body that accepts non-Tensors as input. Got: <class 'tuple'>
[2026-03-28 10:28:42,785] [4/0] torch._dynamo.variables.higher_order_ops: [WARNING] speculate_subgraph: while introspecting the user-defined autograd.Function, we were unable to trace function `trampoline_autograd_fwd` into a single graph. This means that Dynamo was unable to prove safety for this API and will fall back to eager-mode PyTorch, which could lead to a slowdown.
[2026-03-28 10:28:42,786] [4/0] torch._dynamo.variables.higher_order_ops: [ERR

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/data1/lesliec/sarthak/caduceus/.pixi/envs/default/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2882381/3214334826.py", line 2, in <module>
    _ = model(x)
    ^^^^^^^^
  File "/data1/lesliec/sarthak/caduceus/.pixi/envs/default/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1518, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data1/lesliec/sarthak/caduceus/.pixi/envs/default/lib/python3.11/site-packages/torch/nn/modules/module.py", line 1527, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/data1/lesliec/sarthak/caduceus/.pixi/envs/default/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py", line 328, in _fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/

In [7]:
#yeah compile won't work. But let's see if we do fused mlp and stuff if it speeds it up
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped')
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

10.978900594636798 ms


## redo testing

In [2]:
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped')
# model = torch.compile(model)
model.cuda().bfloat16()

StripedMambaBackbone(
  (hydra_blocks): ModuleList(
    (0-3): 4 x HydraBlock(
      (mixer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mixer): Hydra(
        (in_proj): Linear(in_features=64, out_features=516, bias=False)
        (conv1d): Conv1d(384, 384, kernel_size=(7,), stride=(1,), padding=(3,), groups=384)
        (act): SiLU()
        (fc_D): Linear(in_features=128, out_features=2, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=128, out_features=64, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
      (mlp_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (mlp): MLP(
        (fc1): Linear(in_features=64, out_features=256, bias=True)
        (fc2): Linear(in_features=256, out_features=64, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (act): GELU(approximate='none')
      )
      (mlp_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleL

In [ ]:
x = torch.randn(2, 1024, 64, device='cuda', dtype=torch.bfloat16)
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped')
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):                                                                                               
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#actually 10.4

1040.5569083988667 ms


In [ ]:
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', residual_in_fp32=True)
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#actually 11.16
#so minor slowdown but not consequenctial!

1115.7156787114218 ms


In [ ]:
#let's test gating
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', use_gating=False)
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")
#slightly slower??

1130.06150519941 ms


In [ ]:
#make it more alphagenome like
#wait we not even doing pooling between them lol
#huh why this slowdown?
#this makes no sense since we literally didn't do pooling here...
#was 2000 last time, now 10.8, I think just random speedup/slowdown lol
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', use_gating=True, sampling_use_weight_std=True)
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

1086.449021892622 ms


In [3]:
#let's test if we do some pooling!
x = torch.randn(2, 1024*4, 64, device='cuda', dtype=torch.bfloat16)
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', use_gating=True, sampling_use_weight_std=True, global_pooling=2, transformer_pooling=2)
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(1000):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

1681.5854889107868 ms


In [ ]:
#that's the base, pretty consistent
#fused mlp actually cannot be done because have to recompile my entire codebase with cutlass thing with flash attention...
#it's fine just don't do it
#between 12 and 30 ms wtf lol depends on run but seems minimally additional

model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', residual_in_fp32=True)
model.cuda().bfloat16()
for _ in range(3):                                                                                                                                              
    _ = model(x)                                                                                                                                                
                
torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):                                                                                                                                             
    _ = model(x)
torch.cuda.synchronize()                                                                                                                                        
print((time.perf_counter() - t0) / 10 * 1000, "ms")

26.5550693962723 ms


In [7]:
#let's take an input then, B x L, x D_model
model = StripedMambaBackbone(d_model=64, n_blocks=4, head_dim=4, mode='striped', residual_in_fp32=True)
model.cuda().bfloat16()
test_input = torch.randn(2, 128, 64).to(torch.bfloat16)
output = model(test_input.cuda())
output.shape

torch.Size([2, 128, 64])

In [8]:
output = model(test_input.cuda())
#15 s the first time, almost instant the next

## benchmark vs caduceus

In [1]:
#first we need to load in the right thing, similar config etc.
#also made some more minor changes to allow for no MLP after hydra
import torch
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone

'''here's a config
model:
  _name_: dna_embedding_caduceus
  config:
    _target_: caduceus.configuration_caduceus.CaduceusConfig
    d_model: 256
    n_layer: 16
    vocab_size: 1
    ssm_cfg:
      d_state: 16
      d_conv: 4
      expand: 2
      dt_rank: auto
      dt_min: 0.001
      dt_max: 0.1
      dt_init: random
      dt_scale: 1.0
      dt_init_floor: 0.0001
      conv_bias: true
      bias: false
      use_fast_path: true
    rms_norm: true
    fused_add_norm: true
    residual_in_fp32: false
    pad_vocab_size_multiple: 1
    norm_epsilon: 1.0e-05
    initializer_cfg:
      initializer_range: 0.02
      rescale_prenorm_residual: true
      n_residuals_per_layer: 1
    bidirectional: true
    bidirectional_strategy: add
    bidirectional_weight_tie: true
    rcps: false
    complement_map: null
    skip_embedding: true

'''
yaml_path = '/data1/lesliec/sarthak/caduceus/outputs/2026-03-16/18-19-47-711591/.hydra/config.yaml'
import yaml
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['model']['config']

# caduceus_cfg = CaduceusConfig(**cfg['model']['config'])

# self.backbone = DNAEmbeddingModelCaduceus(config=caduceus_cfg)

{'_target_': 'caduceus.configuration_caduceus.CaduceusConfig',
 'd_model': 256,
 'n_layer': 16,
 'vocab_size': 1,
 'ssm_cfg': {'d_state': 16,
  'd_conv': 4,
  'expand': 2,
  'dt_rank': 'auto',
  'dt_min': 0.001,
  'dt_max': 0.1,
  'dt_init': 'random',
  'dt_scale': 1.0,
  'dt_init_floor': 0.0001,
  'conv_bias': True,
  'bias': False,
  'use_fast_path': True},
 'rms_norm': True,
 'fused_add_norm': True,
 'residual_in_fp32': False,
 'pad_vocab_size_multiple': 1,
 'norm_epsilon': 1e-05,
 'initializer_cfg': {'initializer_range': 0.02,
  'rescale_prenorm_residual': True,
  'n_residuals_per_layer': 1},
 'bidirectional': True,
 'bidirectional_strategy': 'add',
 'bidirectional_weight_tie': True,
 'rcps': False,
 'complement_map': None,
 'skip_embedding': True}

In [2]:
from caduceus.configuration_caduceus import CaduceusConfig
cfg['model']['config']
cfg['model']['config'].pop('_target_')
caduceus_cfg = CaduceusConfig(**cfg['model']['config'])

In [3]:
caduceus_cfg

CaduceusConfig {
  "bidirectional": true,
  "bidirectional_strategy": "add",
  "bidirectional_weight_tie": true,
  "cnn_embedding": false,
  "cnn_embedding_dim": 4,
  "complement_map": {
    "0": 0,
    "1": 1,
    "10": 7,
    "11": 11,
    "12": 12,
    "13": 13,
    "14": 14,
    "15": 15,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 10,
    "8": 9,
    "9": 8
  },
  "d_model": 256,
  "fused_add_norm": true,
  "initializer_cfg": {
    "initializer_range": 0.02,
    "n_residuals_per_layer": 1,
    "rescale_prenorm_residual": true
  },
  "model_type": "caduceus",
  "n_layer": 16,
  "norm_epsilon": 1e-05,
  "pad_vocab_size_multiple": 1,
  "rcps": false,
  "residual_in_fp32": false,
  "rms_norm": true,
  "skip_embedding": true,
  "ssm_cfg": {
    "bias": false,
    "conv_bias": true,
    "d_conv": 4,
    "d_state": 16,
    "dt_init": "random",
    "dt_init_floor": 0.0001,
    "dt_max": 0.1,
    "dt_min": 0.001,
    "dt_rank": "auto",
    "dt_scale": 1.0,
    "exp

In [4]:
from src.models.sequence.dna_embedding import DNAEmbeddingModelCaduceus
caduceus_model = DNAEmbeddingModelCaduceus(config=caduceus_cfg)

In [5]:
caduceus_model

DNAEmbeddingModelCaduceus(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): Transpose()
      )
      (layers): ModuleList(
        (0-15): 16 x Block(
          (norm): RMSNorm()
          (mixer): BiMambaWrapper(
            (mamba_fwd): Mamba(
              (in_proj): Linear(in_features=256, out_features=1024, bias=False)
              (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
              (act): SiLU()
              (x_proj): Linear(in_features=512, out_features=48, bias=False)
              (dt_proj): Linear(in_features=16, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=256, bias=False)
            )
            (mamba_rev): Mamba(
              (in_proj): Linear(in_features=256, out_features=1024, bias=False)
              (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
           

In [6]:
  # ┌────────────────────────┬────────────────────────┬────────────────────────────────────┐
  # │        Caduceus        │       Your model       │               Notes                │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ d_model=256            │ d_model=256            │ direct match                       │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ n_layer=16             │ n_blocks=16            │ direct match                       │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ d_state=16             │ d_state=16             │ direct match                       │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ d_conv=4               │ d_conv=4               │ Hydra default is 7, set explicitly │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ expand=2               │ expand=2               │ direct match                       │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ rms_norm=True          │ norm='rms'             │ direct match                       │
  # ├────────────────────────┼────────────────────────┼────────────────────────────────────┤
  # │ residual_in_fp32=False │ residual_in_fp32=False │ direct match                       │
  # └────────────────────────┴────────────────────────┴────────────────────────────────────┘

In [7]:
#let's now get my model similar
model = StripedMambaBackbone(d_model=caduceus_cfg.d_model, n_blocks=caduceus_cfg.n_layer, mode='mamba_only',
                             d_state=caduceus_cfg.ssm_cfg['d_state'], d_conv=caduceus_cfg.ssm_cfg['d_conv'], expand=caduceus_cfg.ssm_cfg['expand'],
                             norm='rms', residual_in_fp32=caduceus_cfg.residual_in_fp32, rescale_prenorm_residual=True)

In [8]:
model

StripedMambaBackbone(
  (hydra_blocks): ModuleList(
    (0-15): 16 x HydraBlock(
      (mixer_norm): RMSNorm()
      (mixer): Hydra(
        (in_proj): Linear(in_features=256, out_features=1104, bias=False)
        (conv1d): Conv1d(576, 576, kernel_size=(4,), stride=(1,), padding=(2,), groups=576)
        (act): SiLU()
        (fc_D): Linear(in_features=512, out_features=8, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=512, out_features=256, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleList()
  (trans_down): ModuleList()
  (trans_up): ModuleList()
  (final_norm): RMSNorm()
)

In [ ]:
#first print number of params
print(sum(p.numel() for p in model.parameters()))
print(sum(p.numel() for p in caduceus_model.parameters()))

6743680
7721216


In [ ]:
#remember, caduceus shares 
def count_parameters(model):
    # This set stores the memory ID of parameters we've already counted
    seen = set()
    total_params = 0
    
    for p in model.parameters():
        if p.data_ptr() not in seen:
            seen.add(p.data_ptr())
            total_params += p.numel()
            
    return total_params

print(f"Unique parameters: {count_parameters(model):,}")
print(f"Unique parameters in Caduceus model: {count_parameters(caduceus_model):,}")
#allegedly model.parameters deduplicates by default lol

Unique parameters: 6,743,680
Unique parameters in Caduceus model: 7,721,216


In [ ]:
#now let's see outputs
device = 'cuda'
dtype = torch.bfloat16
                                                                                                            
B, L = 1, 524288
d_model = caduceus_cfg.d_model
model = model.cuda().to(dtype)
caduceus_model = caduceus_model.cuda().to(dtype)

x = torch.randn(B, L, d_model, device=device, dtype=dtype)
# out2 = caduceus_model(x.transpose(1,2))
# print(x.shape, out2.shape)
out1 = model(x) #can't run with this size, oof!
# print(x.shape, out1.shape, out2.shape)

OutOfMemoryError: CUDA out of memory. Tried to allocate 512.00 MiB. GPU 0 has a total capacty of 79.25 GiB of which 119.88 MiB is free. Including non-PyTorch memory, this process has 79.13 GiB memory in use. Of the allocated memory 77.02 GiB is allocated by PyTorch, and 1.61 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
out2

(tensor([[[-0.6094, -0.9922, -1.7578,  ...,  0.9766, -1.1562, -2.0781],
          [ 0.0762,  1.3750, -0.3301,  ..., -0.5742, -2.5156,  0.7656],
          [-0.9492, -1.0859, -1.4453,  ...,  0.3125,  0.2471,  1.7891],
          ...,
          [ 1.9219, -1.1406,  1.6406,  ...,  0.0210,  1.1641, -0.1001],
          [-1.0703,  1.2266, -1.3906,  ..., -0.5859, -0.3398,  0.8203],
          [-0.2969,  0.1348, -1.2266,  ...,  0.7812,  1.1250, -0.8125]]],
        device='cuda:0', dtype=torch.bfloat16, grad_fn=<LayerNormFnBackward>),
 None)

In [ ]:
out2[0].shape

torch.Size([1, 524288, 256])

In [ ]:
out1

AssertionError: 

# make it smaller

In [1]:
#let's half it and run again
import torch
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone

yaml_path = '/data1/lesliec/sarthak/caduceus/outputs/2026-03-16/18-19-47-711591/.hydra/config.yaml'
import yaml
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)

from caduceus.configuration_caduceus import CaduceusConfig
cfg['model']['config']
cfg['model']['config'].pop('_target_')
caduceus_cfg = CaduceusConfig(**cfg['model']['config'])
from src.models.sequence.dna_embedding import DNAEmbeddingModelCaduceus
caduceus_model = DNAEmbeddingModelCaduceus(config=caduceus_cfg)

#let's now get my model similar
model = StripedMambaBackbone(d_model=caduceus_cfg.d_model, n_blocks=caduceus_cfg.n_layer, mode='mamba_only',
                             d_state=caduceus_cfg.ssm_cfg['d_state'], d_conv=caduceus_cfg.ssm_cfg['d_conv'], expand=caduceus_cfg.ssm_cfg['expand'],
                             norm='rms', residual_in_fp32=caduceus_cfg.residual_in_fp32, rescale_prenorm_residual=True)

device = 'cuda'
dtype = torch.bfloat16
model = model.cuda().to(dtype)
caduceus_model = caduceus_model.cuda().to(dtype)
B, L = 1, 262144
d_model = caduceus_cfg.d_model
x = torch.randn(B, L, d_model, device=device, dtype=dtype)

with torch.no_grad():
    out1 = model(x)
    out2 = caduceus_model(x.transpose(1,2))
print(x.shape, out1.shape, out2[0].shape)

torch.Size([1, 262144, 256]) torch.Size([1, 262144, 256]) torch.Size([1, 262144, 256])


In [2]:
out1.shape


torch.Size([1, 262144, 256])

In [ ]:
#we can benchmark the speed between the hydra and caduceus by doing a simple forward backward test!
import gc
def benchmark(model, x, n_warmup=5, n_runs=50, forward_only=True):                                         
    gc.collect()
    torch.cuda.empty_cache()
    for _ in range(n_warmup):
        if forward_only:                                                                                   
            with torch.no_grad():                                                                          
                model(x)
        else:                                                                                              
            out = model(x)
            if isinstance(out, tuple):
                out = out[0]
            out.sum().backward()
    torch.cuda.synchronize()
                                                                                                            
    t0 = time.perf_counter()
    for _ in range(n_runs):                                                                                
        if forward_only:
            with torch.no_grad():
                model(x)                                                                                   
        else:
            out = model(x)
            if isinstance(out, tuple):
                out = out[0]
            out.sum().backward()                                                                      
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_runs * 1000  # ms
                                                                                                            
print("Forward only:")                                                                                     
print(f"  Old: {benchmark(caduceus_model, x.transpose(1,2)):.2f} ms")                                                          
print(f"  New: {benchmark(model, x):.2f} ms")                                                          
                                                                                                            
#2.3x slower on inference is crazy! Maybe the fused parts? maybe should rewrite? idk what else it could be

Forward only:
  Old: 296.21 ms
  New: 704.03 ms


In [7]:
isinstance(out1,tuple)

False

In [12]:
print("Forward + backward:")
print(f"  Old: {benchmark(caduceus_model, x.transpose(1,2), forward_only=False):.2f} ms")
#much less memory too!

Forward + backward:
  Old: 1046.84 ms


In [13]:
#let's gc collect
import gc

# 1. Clear existing junk
torch.cuda.empty_cache()
gc.collect()

2214

In [14]:
#now let's rerun on new model. Notable increase in memory footprint!
print(f" New: {benchmark(model, x, forward_only=False):.2f} ms")
#oh this is notably slower! 

 New: 1999.75 ms


In [ ]:
#allegedly mamba2 is better at higher dstate, so let's try that
torch.cuda.empty_cache()
gc.collect()

caduceus_cfg.ssm_cfg['d_state'] = 64

caduceus_model = DNAEmbeddingModelCaduceus(config=caduceus_cfg)

#let's now get my model similar
model = StripedMambaBackbone(d_model=caduceus_cfg.d_model, n_blocks=caduceus_cfg.n_layer, mode='mamba_only',
                             d_state=caduceus_cfg.ssm_cfg['d_state'], d_conv=caduceus_cfg.ssm_cfg['d_conv'], expand=caduceus_cfg.ssm_cfg['expand'],
                             norm='rms', residual_in_fp32=caduceus_cfg.residual_in_fp32, rescale_prenorm_residual=True)

model.cuda().bfloat16()
caduceus_model.cuda().bfloat16()

L = 131072
x = torch.randn(B, L, d_model, device=device, dtype=dtype)
print("Forward only:")
print(f"  Old: {benchmark(caduceus_model, x.transpose(1,2)):.2f} ms")                                                          
print(f"  New: {benchmark(model, x):.2f} ms")
print("Forward + backward:")
print(f"  Old: {benchmark(caduceus_model, x.transpose(1,2), forward_only=False):.2f} ms")
print(f"  New: {benchmark(model, x, forward_only=False):.2f} ms")

#ahhhh here we go! Here have to cut length in half and still 20GB compared to 30 when dstate was 16 but length was double
#and new model still 33 GB!


Forward only:
  Old: 418.01 ms
  New: 298.78 ms
Forward + backward:
  Old: 1537.56 ms
  New: 958.54 ms


In [ ]:
#   ┌────────────────────────────┬──────────────────────────────┬────────────┬────────────┐                                                                         
#   │            Goal            │           Pooling            │  d_state   │    Mode    │
#   ├────────────────────────────┼──────────────────────────────┼────────────┼────────────┤                                                                         
#   │ Long-range (Enformer-like) │ High (8-32x global)          │ 64 is fine │ striped    │
#   ├────────────────────────────┼──────────────────────────────┼────────────┼────────────┤
#   │ Local/regional patterns    │ Low (2-4x)                   │ 64-128     │ mamba_only │                                                                         
#   ├────────────────────────────┼──────────────────────────────┼────────────┼────────────┤                                                                         
#   │ Balanced                   │ 8x global + 2-4x transformer │ 64         │ striped    │                                                                         
#   └────────────────────────────┴──────────────────────────────┴────────────┴────────────┘           

#so maybe starting with dstate 64 and doing pooling/striped is good?        

# Bilby vs Caduceus Striped Mamba: Architecture Comparison

## High-level stripe pattern

| | **Bilby** | **Caduceus** |
|---|---|---|
| Stripe ratio | N mamba blocks : 1 attention (3:1 default) | 1:1 (HydraBlock then TransformerBlock per loop iteration) |
| Loop structure | Outer: `transformer_layers`; inner: `mamba_layers` SSM blocks | Single loop over `n_blocks`, each has one hydra + one transformer |

**Fundamental difference**: Bilby groups multiple SSM blocks between each attention block. Caduceus always interleaves 1:1. The 3:1 ratio is common in recent architectures (e.g. Jamba) since attention is more expensive.

---

## SSM block internals

Both use **pre-norm residual** with optional MLP. The key difference is the SSM kernel itself:

- **Bilby**: Standard Mamba-1 style SSM — shift conv (depthwise, k=3) → selective scan with B/C projections. Forward + reverse runs separately, concatenated, then projected down. Gating applied element-wise after SSM.
- **Caduceus (Hydra)**: Mamba-2 SSD-based bidirectional SSM. Different state space formulation (structured, head-grouped via `headdim`/`ngroups`), likely more efficient.

Same spirit (bidirectional SSM), different kernel generation (Mamba1 vs Mamba2 SSD).

---

## Pooling / down+upsample

| | **Bilby** | **Caduceus** |
|---|---|---|
| Global pool | None in backbone (trunk does ~32x CNN pooling before backbone) | `global_down` / `global_up` wrapping entire block stack |
| Per-attention pool | `max_pool(trans_pool_size=4)` → attention → `UNet` repeat-upsample + skip add + depthwise conv | `DownsampleStack` → transformer → `UpsampleStack` per block |
| Skip connections | Pre-pool activations stored and added via simple repeat-interleave + add + depthwise sep conv | Multi-stage conv blocks (norm→GELU→conv) with residual pad/crop |
| Channel variation | Constant `mamba_features` throughout | Optional channel scaling during up/downsample (grow toward `d_model`) |
| Upsample method | `repeat` | `repeat_interleave` + learnable `residual_scale` |

The UNet mechanics are **conceptually identical** (max-pool down, repeat-interleave up, skip connection from pre-pool features). Caduceus's implementation is richer: multi-conv-block stages, channel growth, learnable residual scale. Bilby's is simpler: `x = repeat(x) + u` + one depthwise conv.

---

## Transformer / attention block

Both: pre-norm → MHA → residual → pre-norm → MLP → residual.

- **Bilby**: Enformer-style relative positional biases or RoPE. No explicit gating.
- **Caduceus**: `MHAGate` — adds output gating on top of standard MHA, RoPE optional.

---

## Summary of fundamental differences

1. **SSM ratio**: Bilby uses multiple SSM blocks per attention (3:1 default); Caduceus is 1:1.
2. **SSM kernel**: Mamba-1 style (bilby) vs Mamba-2/Hydra (Caduceus) — different computational structure.
3. **Upsample complexity**: Bilby's UNet is minimal (1 repeat + 1 depthwise conv); Caduceus has a full multi-stage conv tower with norm, GELU, channel ramp, and learnable scale.
4. **Global pooling**: Caduceus has a separate `global_down/up` wrapping the entire backbone; bilby handles that in the CNN trunk before the backbone.
5. **MHA gating**: Caduceus attention has explicit output gating; bilby's doesn't.

The core **striped** concept is the same: alternate SSM and attention, pool for attention, upsample back with skip connections. The details differ in SSM version, stripe ratio, and up/downsampling complexity.


In [ ]:
#I implemented the enformer positional encoding (claude did I need to verify)
#but it doesn't work with flash attention, so maybe not ideal
#But we have the option if it actually did implement it correctly
#

# steps to truly compare

1. compare backbone with caduceus
2. compare cbackbone with enigma
3. compare backbone with bilby
4. compare up and down sampling with alphagenome code

# comparing with caduceus main backbone

In [1]:
#so the config we will be using is here
cfg_path = '/data1/lesliec/sarthak/caduceus/outputs/2026-03-16/18-19-47-711591/.hydra/config.yaml'
#and looking at the code, just calls caduceus with the config from here. /data1/lesliec/sarthak/caduceus/src/models/sequence/dna_embedding.py
#it inherits from nn.Module and GenerationMixin which does decodingstrategies and stuff, but isn't really needed for us, so we can safely ignore it
#then runs its own init
#so basically jsut look into Caduceus class which is here /data1/lesliec/sarthak/caduceus/caduceus/modeling_caduceus.py
#this is what we compare to our real backbone!

In [ ]:
#so base class is PretrainedModel which takes in a bunch of stuff like letting you load hugging face models
#then caduceus pretrained model only thing we need to care about is how it initializes weights
#it does this 0 setting for biases and normalizes embeddings. I see no reason why we can't take those
#it also does this rescale_prenorm_residual. This seems important? I think we might have this already
#we have rescale prenorm residual which is good, but since we do prenorm, the embedding wont' matter, and sicne we do CNN, the bias won't matter either
#I think it's fine as is

#now we get to the meat
#can return just hidden states or all hidden states which comes from caduceus mixer model. all hidden states is just the hidden states from the middle layers of the model
#I guess if you need it, but who cares, we can manually run and extract lol

#simple nn.module (thank god)
#we did fused add norm with rms norm. We do default norm, would have to change it to fused normalization for maybe more speed

#so a few key notes. We have MLP after Hydra layer as optional leaning on no. Make sure caduceus does that
#also we do norm, mixer, mixer_dropout then mlp norm and add residual, mlp, mlp dropout and add residual for Hydra! 
#residual always input to caduceus layer, have to look into the block. Then at end, add residual, norm and return
#we don't have a cross block residual for striped because we can't with CNN breaking it by pooling
#we also don't have to do residual plus norm at th eend since we start blocks with it, they end blocks with it!

#now look at how their blocks are made and how they differ
#Uses Block, which takes in d_model, mxier_cls, nn.Identity, norm_cls, fused_add_norm true vs false, and residual in fp32
#is a mamba ssm block
#norm class is rms norm, mixer class is a bimamba wrapper? which just runs bidirectional mamba by importing mamba
#the Block is very simple, adds residual to hidden state, does norm, runs mixer (bimamba in this case)
#if mlp, add residual, do norm, then run MLP if you want to. And we show that we run nn.IOdentity for mlp class
#we however, do a different approach
# Mamba-style (across blocks):                                                                                                                                                                                                                                                                                                                    
  #   block N:   residual = prev_hs + residual; hs = mixer(norm(residual))                                                                                                                                                                                                                                                                          
  #   block N+1: residual = hs + residual; ...                                                                                                                                                                                                                                                                                                      
                                                                                                                                                                                                                                                                                                                                                    
  # Your style (self-contained):                                                                                                                                                                                                                                                                                                                    
  #   block N:   x = x + mixer(norm(x))                                                                                                                                                                                                                                                                                                             
  #   block N+1: x = x + mixer(norm(x))                                                                                                                                                                                                                                                                                                             
                                                                                                                                                                                                                                                                                                                                                    
  #The only practical difference is what we discussed before: Mamba's style makes it easy to keep the running residual in fp32 across block boundaries (since it's an explicit tensor you control), whereas yours casts back to model dtype at each block exit.
  #but this also doesn't really matter because we only caree about cross block residual if we're doing mamba only which we can and now we have the option
  #basically we have the option jsut less clean code, their implementation allows it but slightly changes the order

#otherwise I think it seems fine! We just run hydra directly isntead
#only need to revisit it if the results are ay off or weird or something, but I think it seems pretty good?

#key ideas
#make sure cross block fp32 is true if doing mamba only. actually can be false, it is in mamba model
#make sure rms norm #Done


# enigma comparson

In [ ]:
#comparison to enigma

#we will look at stuff like MLP activations (probably silu)
#also make sure block structur emakessense, but the overall MHA already from them
#enigma has nice utilities like predicting forward and backward then averaging, but we can just do that in our evals class!
#it does gelu tanh for efficiency, gelu is fine?

#rope base is 10k, and splits the 1536 dim of enformer inso 192 in 8 heads. How do we do that? What's the default supposed to be?
#I infer num heads from head dim and model dim
#20k default for rope base is probalby fine too
#he does mlp and attention dropout and this post mlp/post attention dropout too! and qk norm
#also has a bp resolution model?

#and we get an idea of the kernel size used in unet, but will steal from alpha genome instead

#decoder stuff
#droppout in final head layer too! #TODO
#and importantly they don't do weight dcay in the head. WE should add this to decoder! #TODO
#overall it seems fine, let's check transformer module
#seems perfeclty fine, norm attention, dropout add. And in there they do some dropout in MHA and MLP as well

#idk why they do so much dropout, but maybe because they have a smaller dataset
#Would maybe help for us as well... we should add it at like 0.2 everywhere? especially since have more params?

#so we'll add a post attention/mamba dropout that's before we add the residual
#no need for post mlp dropout, in the mlp itself is probably fine. That can make larger like 0.2 or 0.3 or something and 0.1 post attention/mamba too
#wait we already have it, just set dropout to 0.2 or something? can play with it! #TODO

#otherwise I think this is probably fine and mostly the same!

# bilby comparison

In [ ]:
#we use layer norm for unet
#they also do dropout in transformer

#lots of arguments and thigns we could take from there, but also lots of differences like no head dropout, so idk man. Dropout in transformer tho?
#allegedly enformer positional basis is the same. Still want to verify it before I use it...

#otherwise I think we're honestly fine?
#the rest of it looks fine, lots of hyperparameters to tune I guess? We can explore a lot
#TODO look at MHA gating and see if it does anything!

# UNET comparison

In [ ]:
#the actual hard part, have to really read and understand their pseudocode!
#first just look through their pseudocode, no reading online!

#convolutions use same padding
#conv block is RMSBatchNorm, Gelu, StandardizedConv1d (width 5). 
#firstmake sure width is 5 default in downsample (it is)
#The RMSBatchNorm has Each RMSBatchNorm layer maintains an exponential moving average (EMA, decay 0.9) of per-channel variance during training; this EMA variance is used for normalization during inference
#so this part is hard, but basically we can implement it. For batch size 1 use layer norm, for larger ones can use their rms batch norm. 
#fixed it, now it does rms batchnorm too by default, fine for large sequences? maybe sus as sequence gets smaller?

#it also has to have residual skip connections between conv blocks, pad with 0
#so each conv block itself is norm, gelu, standardized conv1d. we indeed have that

#is standardized conv1d right?
#it's very strange, it's actual conv, but update parameters on the fly lol

#they have a dna embedder which is our encoder. We can think of it as having run the first CNN. Then we need to run a conv block for the first time and then add the skip connection
#changed encoder, but make sure that in myy model we first run a conv block if we are downsampling?
#ok default relu, for our models where we do downsampling, then can do none if we run this model because conv block #done and yes. make sure model defaults to no activation in encoder and norm none too. and set transpose true
#ok now we added conv block at beginning to better align. I think this is great? 

#now let's see if we did the skip connections
#so now by default does dna embedder basically but without the conv tower with downsampling unless we do global downsample
#I like this a lot! even my non downsample one gets a few convs with a nice order of normalizations!

#ok then looked into order and it seems fine, start with post norm, transition to prenorm just like we do!
#now let's l ook at the hard part, the down and up sampling lol
#let's see the downsres block, it does a conv block and adds dimensions to the output. Then it pads it so we can add back to the original. Then it does another conv block same shape but keeps residual adding
#let's see what we do. looks literally the same, our padding is a bit different, we right pad, is that what they do? yeah seems right?
#and in terms of the pooling, they store intermediate and max pool
#we also seem to do exactly the same where we max pool and keep the stored intermediates

#so the encoder seems fine, now let's check decoder!

#decoder calculates the reverse order of channels, very simple
#then goes in reverse order
#this is actually pretty complicated and I think it messed up with the intial in samples but maybe the other samples as well? Will play around with it now
#wait somehow d_model and decode_channels[0] are the same?? Claude thinks it's right as is...
#wait no it is, but downsample stack in channels is wrong?
#really need to think about this wtf


#oh here's my issue lol, we actually ahve a bottle neck and stop at d_model again!
#we'll input start_channels 768, d_model 1536, that seems fine? then grow should be calculated directly
#yeah grow channels is calculated properly
#ok so I think changing in_ch is fine?

#ok so we have one key issue, even tho we ran dna embedder, we didn't downsample from it. This means we have an extra convolution operation. This is honestly probably fine?
#I think the biggest issue is we need to start with downsample block and make it do the pooling operation first, then only do 6 layers. So we'll completley redo this later lol.

#actually shouldn't be too bad, can have claude fix this! start with the save intermediate and do pooling.
#IT should first save intermediate and manually do a pool. Then run through loop, also do 1 less layer only in downsample!
#idk why claude is failing on this lol. Feels pretty easy?
#just save to intermediates, do a pool, then let it loop
#check it with claude

#ok I think we fixed it, now we saved a downsample block, let's see if it goes thorugh the upsample block propelry
#let's first make sure the intermediate shapes perfectly match for the downsample block
#first one should be og shape. yes it is
#then we pool, increase channels by 128, so each following one should be half length and 128 more channels
#yeah that looks perfectly right...
#first upres block actually shouldn't increase channel dim, need channels to match to do upsample
#so we do the conv block and then we add residual with crop
#very weird but it does look right...

#it does even work. We'll review the logic once more, but then I legit think we are good??

#if do pooling have to make sure d_in is specified and is what the encoder gives #TODO





In [ ]:
#make sure I use AdamW #done and yes

# other notes

In [ ]:
#seems like we can lower expand to 1 and ddstate to 64 and should be more comparable mamba 2 to 1? Let's hope it fits in memory??


#first we need to load in the right thing, similar config etc.
#also made some more minor changes to allow for no MLP after hydra
import torch
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone

'''here's a config
model:
  _name_: dna_embedding_caduceus
  config:
    _target_: caduceus.configuration_caduceus.CaduceusConfig
    d_model: 256
    n_layer: 16
    vocab_size: 1
    ssm_cfg:
      d_state: 16
      d_conv: 4
      expand: 2
      dt_rank: auto
      dt_min: 0.001
      dt_max: 0.1
      dt_init: random
      dt_scale: 1.0
      dt_init_floor: 0.0001
      conv_bias: true
      bias: false
      use_fast_path: true
    rms_norm: true
    fused_add_norm: true
    residual_in_fp32: false
    pad_vocab_size_multiple: 1
    norm_epsilon: 1.0e-05
    initializer_cfg:
      initializer_range: 0.02
      rescale_prenorm_residual: true
      n_residuals_per_layer: 1
    bidirectional: true
    bidirectional_strategy: add
    bidirectional_weight_tie: true
    rcps: false
    complement_map: null
    skip_embedding: true

'''
yaml_path = '/data1/lesliec/sarthak/caduceus/outputs/2026-03-16/18-19-47-711591/.hydra/config.yaml'
import yaml
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['model']['config']

# caduceus_cfg = CaduceusConfig(**cfg['model']['config'])

# self.backbone = DNAEmbeddingModelCaduceus(config=caduceus_cfg)
from caduceus.configuration_caduceus import CaduceusConfig
cfg['model']['config']
cfg['model']['config'].pop('_target_')
caduceus_cfg = CaduceusConfig(**cfg['model']['config'])
caduceus_cfg
from src.models.sequence.dna_embedding import DNAEmbeddingModelCaduceus
caduceus_model = DNAEmbeddingModelCaduceus(config=caduceus_cfg)
model = StripedMambaBackbone(d_model=caduceus_cfg.d_model, n_blocks=caduceus_cfg.n_layer, mode='mamba_only',
                             d_state=64, d_conv=caduceus_cfg.ssm_cfg['d_conv'], expand=1,
                             norm='rms', residual_in_fp32=caduceus_cfg.residual_in_fp32, rescale_prenorm_residual=True)


# model = torch.compile(model)
model.cuda().bfloat16()

StripedMambaBackbone(
  (hydra_blocks): ModuleList(
    (0-15): 16 x HydraBlock(
      (mixer_norm): RMSNorm()
      (mixer): Hydra(
        (in_proj): Linear(in_features=256, out_features=776, bias=False)
        (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(2,), groups=512)
        (act): SiLU()
        (fc_D): Linear(in_features=256, out_features=4, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleList()
  (trans_down): ModuleList()
  (trans_up): ModuleList()
  (final_norm): RMSNorm()
)

In [ ]:
model.hydra_blocks[0].mixer.in_proj.weight #is bfloat 16 on cuda!

Parameter containing:
tensor([[ 0.0447,  0.0223, -0.0354,  ..., -0.0153, -0.0143,  0.0457],
        [-0.0457, -0.0525,  0.0503,  ...,  0.0095,  0.0596, -0.0187],
        [-0.0378, -0.0325, -0.0315,  ..., -0.0491,  0.0271,  0.0192],
        ...,
        [ 0.0016,  0.0527, -0.0449,  ...,  0.0003,  0.0469,  0.0126],
        [ 0.0442, -0.0309, -0.0282,  ...,  0.0469, -0.0132, -0.0603],
        [ 0.0398, -0.0327, -0.0371,  ...,  0.0178,  0.0047,  0.0625]],
       device='cuda:0', dtype=torch.bfloat16, requires_grad=True)

In [3]:
device = 'cuda'
dtype = torch.bfloat16
                                                                                                            
B, L = 1, 524288
d_model = caduceus_cfg.d_model
# model = model.cuda().to(dtype)
# caduceus_model = caduceus_model.cuda().to(dtype)

x = torch.randn(B, L, d_model, device=device, dtype=dtype)
# out2 = caduceus_model(x.transpose(1,2))
# print(x.shape, out2.shape)
out1 = model(x) #can't run with this size, oof!
out1.shape

torch.Size([1, 524288, 256])

In [4]:
a = out1.sum()
a.backward()

In [6]:
#so this does seem to fit in memory (barely) is at 78GB lol.
import gc
torch.cuda.empty_cache()
gc.collect()

207

In [7]:
#let's run a bunch of forward backward passes
for _ in range(3):
    out1 = model(x)
    out1.sum().backward()

In [8]:
#so this does seem to fit in memory (barely) is at 78GB lol.
import gc
torch.cuda.empty_cache()
gc.collect()

0

In [ ]:
#now let's do even more
for _ in range(10):
    out1 = model(x)
    out1.sum().backward()
#peaks at 78 I think?
#issue is with ADAM I think we reach oom. Will have to test it

In [ ]:
#let's now rerun it with caduceus model

#first we need to load in the right thing, similar config etc.
#also made some more minor changes to allow for no MLP after hydra
import torch
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
from caduceus.configuration_caduceus import CaduceusConfig
from src.models.sequence.dna_embedding import DNAEmbeddingModelCaduceus

'''here's a config
model:
  _name_: dna_embedding_caduceus
  config:
    _target_: caduceus.configuration_caduceus.CaduceusConfig
    d_model: 256
    n_layer: 16
    vocab_size: 1
    ssm_cfg:
      d_state: 16
      d_conv: 4
      expand: 2
      dt_rank: auto
      dt_min: 0.001
      dt_max: 0.1
      dt_init: random
      dt_scale: 1.0
      dt_init_floor: 0.0001
      conv_bias: true
      bias: false
      use_fast_path: true
    rms_norm: true
    fused_add_norm: true
    residual_in_fp32: false
    pad_vocab_size_multiple: 1
    norm_epsilon: 1.0e-05
    initializer_cfg:
      initializer_range: 0.02
      rescale_prenorm_residual: true
      n_residuals_per_layer: 1
    bidirectional: true
    bidirectional_strategy: add
    bidirectional_weight_tie: true
    rcps: false
    complement_map: null
    skip_embedding: true

'''
yaml_path = '/data1/lesliec/sarthak/caduceus/outputs/2026-03-16/18-19-47-711591/.hydra/config.yaml'
import yaml
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['model']['config']

# caduceus_cfg = CaduceusConfig(**cfg['model']['config'])

# self.backbone = DNAEmbeddingModelCaduceus(config=caduceus_cfg)
cfg['model']['config']
cfg['model']['config'].pop('_target_')
caduceus_cfg = CaduceusConfig(**cfg['model']['config'])

model_caduceus = DNAEmbeddingModelCaduceus(config=caduceus_cfg)
model_caduceus.cuda().bfloat16()

DNAEmbeddingModelCaduceus(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): Transpose()
      )
      (layers): ModuleList(
        (0-15): 16 x Block(
          (norm): RMSNorm()
          (mixer): BiMambaWrapper(
            (mamba_fwd): Mamba(
              (in_proj): Linear(in_features=256, out_features=1024, bias=False)
              (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
              (act): SiLU()
              (x_proj): Linear(in_features=512, out_features=48, bias=False)
              (dt_proj): Linear(in_features=16, out_features=512, bias=True)
              (out_proj): Linear(in_features=512, out_features=256, bias=False)
            )
            (mamba_rev): Mamba(
              (in_proj): Linear(in_features=256, out_features=1024, bias=False)
              (conv1d): Conv1d(512, 512, kernel_size=(4,), stride=(1,), padding=(3,), groups=512)
           

In [ ]:
#after forward backward, we had 78754MiB for mamba2 based model, let's compare to mamba1 here
B, L = 1, 524288
d_model = caduceus_cfg.d_model
device = 'cuda'
dtype = torch.bfloat16
x = torch.randn(B, L, d_model, device=device, dtype=dtype)
for _ in range(10):
    out2 = model_caduceus(x.transpose(1,2))
    out2[0].sum().backward()

#yeah lol is like 67GB... goes up to 70GB I guess, but it is dangerously close. We can always test and see. Otherwise checkpoint something
#or maybe make dstate 32? woudl that be more fair since expand2*16 = 32 anyways. Maybe that's the best appraoch?

In [7]:
# can look more specificallya t model parameters

#in terms of the  model parameters, can update like this?

'''
model:
  _name_: dna_embedding_hydra
  config:
    _target_: hydra.configuration_hydra.HydraConfig # Update to your specific Hydra target
    d_model: 256
    n_layer: 16
    ssm_cfg:
      d_state: 64             # MODIFIED: Increased from 16 to 64/128
      d_conv: 4
      expand: 1               # MODIFIED: Reduced from 2 to 1
      headdim: 64             # NEW: Multi-head dimension
      ngroups: 1              # NEW: Grouped routing 
      chunk_size: 256         # NEW: Hardware chunk size
      dt_min: 0.001           #hydra default
      dt_max: 0.1             #hydra default
      dt_init_floor: 0.0001   #hydra default
      conv_bias: true         #also defaults
      bias: false             #also defaults
      use_mem_eff_path: true  # REPLACED: was 'use_fast_path'
      # REMOVED: dt_rank
      
    norm: rms # MODIFIED: Set to 'rms' to match your implementation
    fused_add_norm: false # MODIFIED: Set to False to match your implementation
    residual_in_fp32: false # MODIFIED: Set to false to match your implementation
    rescale_prenorm_residual: true
'''

#these are the ideas, let's actually test the forward backward of this model. Can also reduce chunk_size if needed
#have to change d_conv to 5 to make it work with use mem eff path
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=256, n_blocks=16, mode='mamba_only',
                                d_state=64, d_conv=5, expand=1, norm='rms', residual_in_fp32=False, rescale_prenorm_residual=True,
                                ngroups=1, head_dim=64, chunk_size=256, use_mem_eff_path=True, headdim=64)
model.cuda().bfloat16()

StripedMambaBackbone(
  (conv_block): ConvBlock(
    (norm): RMSBatchNorm1d()
    (act): GELU(approximate='none')
    (op): StandardizedConv1d(256, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (hydra_blocks): ModuleList(
    (0-15): 16 x HydraBlock(
      (mixer_norm): RMSNorm()
      (mixer): Hydra(
        (in_proj): Linear(in_features=256, out_features=776, bias=False)
        (conv1d): Conv1d(512, 512, kernel_size=(5,), stride=(1,), padding=(2,), groups=512)
        (act): SiLU()
        (fc_D): Linear(in_features=256, out_features=4, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleList()
  (trans_down): ModuleList()
  (trans_up): ModuleList()
  (final_norm): RMSNorm()
)

In [8]:
import torch
B = 1
L = 524288
d_model = 256
device = 'cuda'
dtype = torch.bfloat16
x = torch.randn(B, L, d_model, device=device, dtype=dtype)
for _ in range(10):
    out1 = model(x)
    out1.sum().backward()

In [9]:
#only like 46 GB now??

#let's see the speed
import gc
import time
torch.cuda.empty_cache()
gc.collect()

t0 = time.perf_counter()
for _ in range(10):
    out1 = model(x)
    out1.sum().backward()
torch.cuda.synchronize()
print(f"Average time per forward+backward: {(time.perf_counter() - t0) / 10 * 1000:.2f} ms")


Average time per forward+backward: 3939.44 ms


In [1]:
#compare this to the non memory effective one
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=256, n_blocks=16, mode='mamba_only',
                                d_state=64, d_conv=5, expand=1, norm='rms', residual_in_fp32=False, rescale_prenorm_residual=True,
                                ngroups=1, chunk_size=256, use_mem_eff_path=False, headdim=64)
model.cuda().bfloat16()

StripedMambaBackbone(
  (conv_block): ConvBlock(
    (norm): RMSBatchNorm1d()
    (act): GELU(approximate='none')
    (op): StandardizedConv1d(256, 256, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (hydra_blocks): ModuleList(
    (0-15): 16 x HydraBlock(
      (mixer_norm): RMSNorm()
      (mixer): Hydra(
        (in_proj): Linear(in_features=256, out_features=776, bias=False)
        (conv1d): Conv1d(512, 512, kernel_size=(5,), stride=(1,), padding=(2,), groups=512)
        (act): SiLU()
        (fc_D): Linear(in_features=256, out_features=4, bias=False)
        (norm): RMSNorm()
        (out_proj): Linear(in_features=256, out_features=256, bias=False)
      )
      (mixer_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (transformer_blocks): ModuleList()
  (trans_down): ModuleList()
  (trans_up): ModuleList()
  (final_norm): RMSNorm()
)

In [ ]:
#warmup first
import torch
import time
B = 1
L = 524288
d_model = 256
device = 'cuda'
dtype = torch.bfloat16
x = torch.randn(B, L, d_model, device=device, dtype=dtype)
for _ in range(3):
    out1 = model(x)
    out1.sum().backward()
torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(10):
    out1 = model(x)
    out1.sum().backward()
torch.cuda.synchronize()
print(f"Average time per forward+backward: {(time.perf_counter() - t0) / 10 * 1000:.2f} ms")
#yeah takes 75GB!!
#a little bit faster, but probalby not worth worrying about

Average time per forward+backward: 3466.91 ms


In [ ]:
#final things, let's see if we can make it larger now that we are doing memory efficiency
import sys
import torch
import time
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=256, n_blocks=16, mode='mamba_only',
                                d_state=64, d_conv=5, expand=2, norm='rms', residual_in_fp32=False, rescale_prenorm_residual=True,
                                ngroups=1, head_dim=64, chunk_size=256, use_mem_eff_path=True, headdim=64)
model.cuda().bfloat16()

B = 1
L = 524288
d_model = 256
device = 'cuda'
dtype = torch.bfloat16
x = torch.randn(B, L, d_model, device=device, dtype=dtype)
for _ in range(3):
    out1 = model(x)
    out1.sum().backward()
torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(10):
    out1 = model(x)
    out1.sum().backward()
torch.cuda.synchronize()
print(f"Average time per forward+backward: {(time.perf_counter() - t0) / 10 * 1000:.2f} ms")
#only 66GB but we can see that it is indeed a bit slower lol. NEarly half the speed, but way more expressive probably? maybe don't need that much expressiveness?
#still for the baseline models why don't we do it?

Average time per forward+backward: 6531.09 ms


In [ ]:
#so let's try training with these parameters, otherwise it's ok? can reduce it and likely more fair of a comparison, this is just testing expressiveness!



# let's test upsample downsample and overall model

In [1]:
#now test the upsample/downsample
import sys
sys.path.append('/data1/lesliec/sarthak/caduceus/')
from src.models.nn.sampling import DownsampleStack, UpsampleStack
import torch
a = DownsampleStack(1536,128)
test = torch.rand((1,4096,1536//2))
out = a(test)
print(out[0].shape)
for i in out[1]:
    print(i.shape)

torch.Size([1, 32, 1536])
torch.Size([1, 4096, 768])
torch.Size([1, 2048, 896])
torch.Size([1, 1024, 1024])
torch.Size([1, 512, 1152])
torch.Size([1, 256, 1280])
torch.Size([1, 128, 1408])
torch.Size([1, 64, 1536])


In [ ]:
b = UpsampleStack(1536,128)
out2 = b(out[0], out[1])
print(out2.shape)
#yeah this seems fine

torch.Size([1, 4096, 768])


In [2]:
#now let's import the main backbone
from src.models.sequence.striped_backbone import StripedMambaBackbone
model = StripedMambaBackbone(d_model=1536, d_in=1536//2, n_blocks=1, mode='mamba_only', d_state=64, d_conv=4, expand=1, norm='rms', residual_in_fp32=False, rescale_prenorm_residual=True, global_pooling=4)
model

StripedMambaBackbone(
  (conv_block): ConvBlock(
    (norm): RMSBatchNorm1d()
    (act): GELU(approximate='none')
    (op): StandardizedConv1d(768, 768, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (global_down): DownsampleStack(
    (layers): ModuleList(
      (0): DownsampleLayer(
        (downres_increase): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(768, 1536, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (downres_refine): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(1536, 1536, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
  )
  (global_up): UpsampleStack(
    (layers): ModuleList(
      (0): UpsampleLayer(
        (reduce_main): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): 

In [ ]:
model = StripedMambaBackbone(d_model=1536, d_in=1536//2, n_blocks=1, mode='striped', d_state=64, d_conv=4, expand=1, norm='rms', residual_in_fp32=False, rescale_prenorm_residual=True, global_pooling=4)
model
#this also seems good!

StripedMambaBackbone(
  (conv_block): ConvBlock(
    (norm): RMSBatchNorm1d()
    (act): GELU(approximate='none')
    (op): StandardizedConv1d(768, 768, kernel_size=(5,), stride=(1,), padding=(2,))
  )
  (global_down): DownsampleStack(
    (layers): ModuleList(
      (0): DownsampleLayer(
        (downres_increase): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(768, 1536, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (downres_refine): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): GELU(approximate='none')
          (op): StandardizedConv1d(1536, 1536, kernel_size=(5,), stride=(1,), padding=(2,))
        )
        (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
  )
  (global_up): UpsampleStack(
    (layers): ModuleList(
      (0): UpsampleLayer(
        (reduce_main): ConvBlock(
          (norm): RMSBatchNorm1d()
          (act): 

In [ ]:
#let's see which arguments we really need for hydra

